In [ ]:
import sys
from pathlib import Path

# Ajustar caminho do projeto
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from datetime import datetime, timedelta
import json
import re
from collections import Counter, defaultdict

from personal_assistant.config import RAW_DATA_DIR, PROCESSED_DATA_DIR
from personal_assistant.dataset import (
    parse_chat_file,
    clean_message_text,
    segment_sessions,
    build_dialogues,
    select_diverse_conversations,
    classify_dialogue_topics,
    save_jsonl,
    TOPIC_KEYWORDS,
)

print(f"Project root: {project_root}")
print(f"Raw data dir: {RAW_DATA_DIR}")
print(f"Processed data dir: {PROCESSED_DATA_DIR}")

## 1. Inspeção dos Dados Brutos (`chat.txt`)

Vamos examinar o arquivo original `data/raw/chat.txt` para entender o formato dos carimbos de data/hora, participantes e primeiras linhas.

In [ ]:
chat_file = RAW_DATA_DIR / "chat.txt"

with open(chat_file, "r", encoding="utf-8") as f:
    raw_lines = f.readlines()

print(f"Total de linhas brutas: {len(raw_lines):,}")
print("\n--- Primeiras 12 linhas do chat bruto ---")
for i, line in enumerate(raw_lines[:12]):
    print(f"L{i+1:02d}: {line.rstrip()}")

## 2. Limpeza de Mensagens e Tratamento de Omissões

No chat bruto, encontramos:
- **Linha 1:** Mensagem de criptografia do WhatsApp (sem remetente).
- **Omissões de Mídia:** `<imagem ocultada>`, `<figurinha omitida>`, `<mensagem de voz omitida>`, etc. Quando há legenda junto da mídia, a legenda é preservada; quando só há a tag de mídia, a mensagem é descartada.
- **Mensagens Apagadas:** `Mensagem apagada` ou `Mensagem apagada por um admin`.
- **Promoções Automáticas:** Cards de links de afiliados (`🔗`, `🏷 Cupom`, `Parcelado 12x`).
- **Continuações Multilinha:** Linhas seguintes sem timestamp são integradas à mensagem anterior.

In [ ]:
full_raw_text = "".join(raw_lines)

media_tags = Counter(re.findall(r"<[^>]+>", full_raw_text))
print("Ocorrências de tags de mídia:")
for tag, count in media_tags.most_common(7):
    print(f"  {tag:<28}: {count:>4} vezes")

deleted_count = len(re.findall(r"mensagem\s+apagada", full_raw_text, re.IGNORECASE))
print(f"\nMensagens apagadas detectadas: {deleted_count}")

### Executando o Parser e a Limpeza

Chamamos `parse_chat_file()`, que executa toda a extração de carimbos, agregação multilinha e filtragem de conteúdo inútil.

In [ ]:
cleaned_messages = parse_chat_file(chat_file)
print(f"Mensagens úteis pós-limpeza: {len(cleaned_messages):,}")

sender_counts = Counter(m["sender"] for m in cleaned_messages)
print("\nDistribuição por remetente:")
for sender, count in sender_counts.most_common():
    pct = (count / len(cleaned_messages)) * 100
    print(f"  {sender:<28}: {count:>5} ({pct:>5.1f}%)")

## 3. Análise de Intervalos de Tempo e Segmentação de Sessões

Para agrupar mensagens em conversas coerentes, calculamos a diferença de tempo ($\Delta t$ em minutos) entre cada par consecutivo de mensagens.

In [ ]:
time_diffs = [
    (cleaned_messages[i]["timestamp"] - cleaned_messages[i-1]["timestamp"]).total_seconds() / 60.0
    for i in range(1, len(cleaned_messages))
]

sorted_diffs = sorted(time_diffs)
n = len(sorted_diffs)
print("Percentis dos intervalos entre mensagens:")
for p in [50, 75, 80, 90, 95]:
    print(f"  Percentil {p:>2}%: {sorted_diffs[int(n * p / 100)]:>6.2f} minutos")

print("\nImpacto de diferentes limiares de sessão:")
for threshold in [15, 20, 30]:
    sess_count = sum(1 for d in time_diffs if d >= threshold) + 1
    print(f"  Limiar de {threshold} min -> {sess_count} sessões de conversa")

> **Conclusão:** O limiar de **20 minutos** agrupa conversas contínuas de forma natural: 90% das trocas ocorrem em menos de 8.5 minutos. Gaps superiores a 20 minutos marcam pausas e transições de contexto.

In [ ]:
sessions = segment_sessions(cleaned_messages, max_gap_minutes=20.0)
print(f"Total de sessões criadas (gap <= 20 min): {len(sessions)}")

lens = [len(s) for s in sessions]
median_l = sorted(lens)[len(lens)//2]
print(f"Mensagens por sessão: Mín={min(lens)}, Mediana={median_l}, Máx={max(lens)}")

## 4. Construção dos Diálogos (`system`, `user`, `assistant`)

Regras de transformação aplicadas:
1. **Mapeamento:** `"Você"` $\rightarrow$ `assistant`. Outros participantes $\rightarrow$ `user`.
2. **Fusão de turnos consecutivos:** Mensagens consecutivas do mesmo papel são unificadas com quebra de linha `\n`.
3. **Início e Fim:** O diálogo obrigatoriamente inicia com `user` e encerra com resposta do `assistant`.
4. **System Prompt:**
   `"Você é o clone digital de Yan Chagas. Responde direto ao ponto e de forma autêntica."`

In [ ]:
dialogues = build_dialogues(
    sessions=sessions,
    assistant_sender="Você",
    clone_name="Yan Chagas",
    max_turns=8
)

print(f"Total de diálogos válidos construídos: {len(dialogues)}")

turn_counts = [len(d["messages"]) - 1 for d in dialogues]
turn_dist = Counter(turn_counts)
print("\nDistribuição de quantidade de turnos (mensagens por diálogo, sem system):")
for turns in sorted(turn_dist.keys()):
    print(f"  {turns:>2} mensagens: {turn_dist[turns]:>3} diálogos")

## 5. Classificação por Tópicos Temáticos

Categorizamos as conversas para garantir variedade temática no fine-tuning.

In [ ]:
topic_dist = defaultdict(int)
for d in dialogues:
    topics = classify_dialogue_topics(d)
    for t in topics:
        topic_dist[t] += 1

print("Distribuição das conversas por tema:")
for topic, count in sorted(topic_dist.items(), key=lambda x: x[1], reverse=True):
    print(f"  {topic:<22}: {count:>3} conversas")

## 6. Seleção Estratificada de 25 Conversas Diversas

Selecionamos **25 conversas de alta qualidade**, equilibrando os temas (Tecnologia, Estudos, Finanças, Jogos, Alimentação, Transporte, Humor) e o formato das conversas (respostas diretas curtas e trocas dinâmicas de múltiplos turnos).

In [ ]:
sample_25 = select_diverse_conversations(dialogues, n=25, seed=42)
print(f"Conversas selecionadas: {len(sample_25)}\n")

print(f"{'#':<3} | {'Tema Principal':<22} | {'Turnos':<6} | {'Prévia do Diálogo':<55}")
print("-" * 92)

for i, d in enumerate(sample_25):
    topics = classify_dialogue_topics(d)
    main_topic = topics[0] if topics else "geral"
    turns = len(d["messages"]) - 1
    u_sample = d["messages"][1]["content"].replace("\n", " ")[:30]
    a_sample = d["messages"][2]["content"].replace("\n", " ")[:30]
    print(f"{i+1:<3} | {main_topic:<22} | {turns:<6} | U: {u_sample}... -> A: {a_sample}...")

### Inspeção Detalhada de Amostras Selecionadas

In [ ]:
examples_to_display = [0, 4, 11]

for idx in examples_to_display:
    d = sample_25[idx]
    topics = classify_dialogue_topics(d)
    print("=" * 75)
    print(f"CONVERSA #{idx+1} | TEMAS: {', '.join(topics).upper()}")
    print("=" * 75)
    for msg in d["messages"]:
        role = msg["role"].upper()
        print(f"[{role}]:\n{msg['content']}\n")

## 7. Exportação e Validação do Formato `.jsonl`

Gravamos tanto o dataset integral quanto a amostra selecionada de 25 conversas em `data/processed/`.

In [ ]:
full_path = PROCESSED_DATA_DIR / "chat_dataset.jsonl"
sample_path = PROCESSED_DATA_DIR / "chat_dataset_sample25.jsonl"

save_jsonl(dialogues, full_path)
save_jsonl(sample_25, sample_path)

print(f"Dataset completo salvo em: {full_path} ({len(dialogues)} diálogos)")
print(f"Amostra de 25 conversas salva em: {sample_path} ({len(sample_25)} diálogos)")

# Validação rigorosa da estrutura JSONL
with open(sample_path, "r", encoding="utf-8") as f:
    loaded = [json.loads(l) for l in f]

assert len(loaded) == 25, f"Esperado 25 diálogos, obtido {len(loaded)}"
for i, item in enumerate(loaded):
    assert "messages" in item, f"Item {i} sem chave 'messages'"
    assert item["messages"][0]["role"] == "system"
    assert item["messages"][1]["role"] == "user"
    assert item["messages"][-1]["role"] == "assistant"

print("\n✅ Validação concluída: Todas as 25 conversas seguem rigorosamente o padrão de diálogo .jsonl!")

## 8. Execução do Pipeline via Linha de Comando (CLI)

Todo este processamento foi modularizado e está disponível no script `personal_assistant/dataset.py`.

Você pode executar o pipeline diretamente pelo terminal:

```bash
# Execução padrão:
python personal_assistant/dataset.py

# Ou via Makefile:
make data

# Ou com argumentos personalizados:
python personal_assistant/dataset.py --max-gap-minutes 20.0 --sample-size 25 --clone-name "Yan Chagas"
```